In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!pip install -q lime scikit-image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 21.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [12]:
def lime_predict(images):
    model.eval()
    tensors = []

    for img in images:
        img = Image.fromarray(img.astype(np.uint8))
        tensor = val_test_transform(img)
        tensors.append(tensor)

    batch = torch.stack(tensors).to(DEVICE)

    with torch.no_grad():
        logits = model(batch)
        probs = torch.softmax(logits, dim=1)

    return probs.cpu().numpy()

In [13]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

In [14]:
explainer = lime_image.LimeImageExplainer()

In [15]:
LIME_SAVE_DIR = "/content/drive/MyDrive/LIME_Explainability"
os.makedirs(LIME_SAVE_DIR, exist_ok=True)

In [ ]:
for cls, samples in images_per_class.items():
    save_cls_dir = os.path.join(LIME_SAVE_DIR, cls)
    os.makedirs(save_cls_dir, exist_ok=True)

    print(f"🔍 LIME Processing class: {cls}")

    for i, (img_path, label) in enumerate(samples):
        img = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        img_np = np.array(img)

        explanation = explainer.explain_instance(
            img_np,
            classifier_fn=lime_predict,
            top_labels=1,
            hide_color=0,
            num_samples=1000   # increase for better stability
        )

        # Get the label that LIME actually explained (the top predicted label by the model)
        explained_label = explanation.top_labels[0]

        temp, mask = explanation.get_image_and_mask(
            explained_label,  # Use the label that LIME explained
            positive_only=True,
            num_features=5,
            hide_rest=False
        )

        lime_image_result = mark_boundaries(temp / 255.0, mask)

        save_path = os.path.join(save_cls_dir, f"lime_{i}.jpg")
        plt.imsave(save_path, lime_image_result)

print(" LIME explainability saved at:", LIME_SAVE_DIR)

🔍 LIME Processing class: Healthy


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

...

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]